In [1]:
import pandas as pd
import sys
import os

# Ép Jupyter về thư mục gốc
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

from src.mining.clustering import perform_topic_clustering, get_representative_reviews
import yaml

# Đọc cấu hình
with open("configs/params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

# Tải dữ liệu ĐÃ LÀM SẠCH từ bước EDA
df_clean = pd.read_csv("data/processed/featured_reviews.csv")
print(f"Dữ liệu sẵn sàng: {df_clean.shape}")

Dữ liệu sẵn sàng: (10000, 40)


In [2]:
# Lấy tham số từ file cấu hình (n_clusters=5, max_features=1000)
k = params['mining']['clustering']['n_clusters']
max_ft = params['mining']['clustering']['max_features_tfidf']

# Thực thi phân cụm
df_clustered, model_kmeans, vectorizer, sil_score = perform_topic_clustering(
    df_clean, text_col='Cleaned_Review', n_clusters=k, max_features=max_ft
)

# Lấy review đại diện cho từng cụm
print("\n=== CÁC ĐÁNH GIÁ ĐẠI DIỆN CHO TỪNG CHỦ ĐỀ ===")
reps = get_representative_reviews(df_clustered, cluster_col='Cluster_Name', review_col='reviews.text')
for cluster, reviews in reps.items():
    print(f"\n[ CHỦ ĐỀ: {cluster} ]")
    for i, rev in enumerate(reviews):
        print(f"  {i+1}. {rev[:150]}...") # Cắt bớt hiển thị cho gọn
        
# Lưu kết quả phân cụm để dành phân tích Insight sau này
df_clustered.to_csv("data/processed/clustered_reviews.csv", index=False)

Bắt đầu Vector hóa TF-IDF với tối đa 1000 đặc trưng...


Đang chạy K-Means với số cụm K = 5...


Điểm Silhouette Score: 0.0045

--- PHÂN TÍCH CHỦ ĐỀ CÁC CỤM ---
Cụm 0: hotel - great - staff - stay - rooms
Cụm 1: breakfast - clean - nice - good - hotel
Cụm 2: stay - hotel - thank - great - time
Cụm 3: french - quarter - orleans - new - hotel
Cụm 4: room - hotel - night - stay - bed

=== CÁC ĐÁNH GIÁ ĐẠI DIỆN CHO TỪNG CHỦ ĐỀ ===

[ CHỦ ĐỀ: hotel - great - staff - stay - rooms ]
  1. The Hyatt is situated right in the city. Its a 5 minute walk to Pike Markets and the shopping district.The staff are really helpful. The rooms are pre...
  2. Really cool spot. Great lobby - the renovation of the lobby and the rooms really makes this place a comfortable and affordable spot in downtown Omaha....

[ CHỦ ĐỀ: room - hotel - night - stay - bed ]
  1. Not a very well kept place. Cockroaches had fun traveling around the room and bathroom. Owner thought it was her right to park in the handicap parking...
  2. I had stayed in this location twice before when on charity work with no real problem, S

In [3]:
from src.mining.association import perform_apriori

# Lấy tham số từ params.yaml (nếu không có thì dùng mặc định)
min_sup = params.get('mining', {}).get('apriori', {}).get('min_support', 0.05)
min_conf = params.get('mining', {}).get('apriori', {}).get('min_confidence', 0.2)

# Chạy Apriori trên tập dữ liệu df_clustered (đã có từ các cell trên)
rules_df = perform_apriori(df_clustered, rating_col='reviews.rating', min_support=min_sup, min_confidence=min_conf)

if rules_df is not None and not rules_df.empty:
    print("\n=== TOP 10 LUẬT DỊCH VỤ ĐI KÈM KHEN/PHÀN NÀN ===")
    display(rules_df[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))
    
    # Lưu luật ra file để đưa vào báo cáo
    rules_df.to_csv("data/processed/association_rules.csv", index=False)
else:
    print("Hãy thử vào configs/params.yaml và giảm min_support xuống 0.01 hoặc 0.02 nhé!")

Đang chuẩn bị dữ liệu giỏ hàng cho Apriori...
Bắt đầu chạy thuật toán Apriori với min_support=0.05...
Đã tìm thấy 99 luật liên quan đến Khen/Phàn nàn.

=== TOP 10 LUẬT DỊCH VỤ ĐI KÈM KHEN/PHÀN NÀN ===


,antecedents,consequents,support,confidence,lift
278,"(has_clean, has_staff)","(has_breakfast, is_Positive)",0.0746,0.382172,1.516556
276,"(has_breakfast, has_staff)","(has_clean, is_Positive)",0.0746,0.441420,1.473365
253,(has_clean),"(has_breakfast, has_room, is_Positive)",0.0760,0.210877,1.466459
128,(has_bed),"(has_room, is_Positive)",0.0695,0.541277,1.440717
281,(has_clean),"(has_breakfast, has_staff, is_Positive)",0.0746,0.206992,1.431482
280,(has_breakfast),"(has_clean, has_staff, is_Positive)",0.0746,0.240801,1.414809
252,(has_breakfast),"(has_room, is_Positive, has_clean)",0.0760,0.245320,1.409073
248,"(has_breakfast, has_clean)","(has_room, is_Positive)",0.0760,0.529248,1.408698
260,"(has_clean, has_location)","(has_room, is_Positive)",0.0582,0.517794,1.378210
246,"(has_breakfast, has_room)","(has_clean, is_Positive)",0.0760,0.410811,1.371198
